# Titanic Pandas 퀴즈 (10문제)

**데이터**: `train.csv` (Kaggle Titanic)  
**목표**: pandas 핵심 문법 — `loc`, `groupby`, 결측치, 문자열, `lambda`, numpy vectorization

> 각 문제 아래 코드 셀에 답을 작성하세요.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
## 문제 01 | 데이터 로드 & 기본 탐색

`train.csv`를 읽고 아래를 출력하세요.

1. `(행 수, 열 수)` shape  
2. `Age`, `Fare` 컬럼의 `describe()`  
3. `dtypes` 중 객체형(object) 컬럼 이름 리스트

In [3]:
print("shape:", df.shape)
display(df[["Age", "Fare"]].describe())
print("object columns:", df.select_dtypes(include=["object", "string"]).columns.tolist())

shape: (891, 12)


,Age,Fare
count,714.000000,891.000000
mean,29.699118,32.204208
std,14.526497,49.693429
min,0.420000,0.000000
25%,20.125000,7.910400
50%,28.000000,14.454200
75%,38.000000,31.000000
max,80.000000,512.329200


object columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


---
## 문제 02 | loc — 단일 조건 필터링

**생존(Survived=1)** 한 승객만 골라 `Name`, `Sex`, `Age`, `Pclass` 컬럼을 조회하세요.

- `loc` 사용 (필수)
- `Age` 오름차순 정렬 후 상위 5행 출력

In [4]:
survived = df.loc[df["Survived"] == 1, ["Name", "Sex", "Age", "Pclass"]]
survived.sort_values("Age").head()

,Name,Sex,Age,Pclass
803,"Thomas, Master. Assad Alexander",male,0.42,3
755,"Hamalainen, Master. Viljo",male,0.67,2
469,"Baclini, Miss. Helene Barbara",female,0.75,3
644,"Baclini, Miss. Eugenie",female,0.75,3
78,"Caldwell, Master. Alden Gates",male,0.83,2


---
## 문제 03 | loc — 복합 조건 필터링

아래 **세 조건을 모두 만족**하는 승객을 `loc`으로 필터링하세요.

| 조건 | 값 |
|------|----|
| Pclass | 1 (1등석) |
| Sex | female |
| Age | 30 이상 |

필터링된 **승객 수**와 `Fare` 평균(소수점 2자리)을 출력하세요.

In [5]:
mask = (df["Pclass"] == 1) & (df["Sex"] == "female") & (df["Age"] >= 30)
filtered = df.loc[mask]

print("승객 수:", len(filtered))
print("Fare 평균:", round(filtered["Fare"].mean(), 2))

승객 수: 55
Fare 평균: 102.01


---
## 문제 04 | loc — 조건 할당

`Fare`가 **50 초과**인 승객 행에 대해, 새 컬럼 `HighFare`를 `True`, 나머지는 `False`로 설정하세요.

- `loc` 조건 할당 사용
- `HighFare == True` 인 행 수 출력

In [6]:
df["HighFare"] = False
df.loc[df["Fare"] > 50, "HighFare"] = True

print("HighFare == True:", df["HighFare"].sum())

HighFare == True: 160


---
## 문제 05 | groupby — 집계

`Pclass`(객실 등급)별로 아래를 구하세요.

| 컬럼 | 내용 |
|------|------|
| passenger_count | 승객 수 |
| survival_rate | 생존률 (`Survived` 평균, 소수점 3자리) |
| avg_fare | 평균 요금 (소수점 2자리) |

- `groupby` + `agg` 사용

In [7]:
pclass_stats = (
    df.groupby("Pclass")
    .agg(
        passenger_count=("PassengerId", "count"),
        survival_rate=("Survived", "mean"),
        avg_fare=("Fare", "mean"),
    )
)
pclass_stats["survival_rate"] = pclass_stats["survival_rate"].round(3)
pclass_stats["avg_fare"] = pclass_stats["avg_fare"].round(2)
pclass_stats

,passenger_count,survival_rate,avg_fare
Pclass,,,
1,216,0.630,84.15
2,184,0.473,20.66
3,491,0.242,13.68


---
## 문제 06 | groupby — 성별 · 승선항구 교차

`Sex`와 `Embarked`를 기준으로 `groupby`하여 승객 수(`count`)와 평균 나이(`mean`)를 구하세요.

- `Age` 결측 행은 제외하고 계산 (`dropna(subset=['Age'])` 활용)
- 결과를 `mean_age` 오름차순 정렬

In [8]:
sex_emb = (
    df.dropna(subset=["Age"])
    .groupby(["Sex", "Embarked"])
    .agg(
        passenger_count=("PassengerId", "count"),
        mean_age=("Age", "mean"),
    )
    .sort_values("mean_age")
)
sex_emb

passenger_count   mean_age
Sex    Embarked                            
female Q                      12  24.291667
       S                     186  27.771505
       C                      61  28.344262
male   S                     368  30.291440
       Q                      16  30.937500
       C                      69  32.998841

---
## 문제 07 | 결측치 처리

1. `Age`, `Cabin`, `Embarked` 컬럼의 **결측치 개수**를 Series로 출력  
2. 원본을 복사한 `df_filled` DataFrame에서  
   - `Age` → **중앙값(median)** 으로 채우기  
   - `Embarked` → **최빈값(mode)** 으로 채우기  
3. 처리 후 `Age`, `Embarked` 결측치가 0인지 확인

In [9]:
missing = df[["Age", "Cabin", "Embarked"]].isna().sum()
print(missing)

df_filled = df.copy()
df_filled["Age"] = df_filled["Age"].fillna(df_filled["Age"].median())
df_filled["Embarked"] = df_filled["Embarked"].fillna(df_filled["Embarked"].mode()[0])

print("\n처리 후 결측:")
print(df_filled[["Age", "Embarked"]].isna().sum())

Age         177
Cabin       687
Embarked      2
dtype: int64

처리 후 결측:
Age         0
Embarked    0
dtype: int64


---
## 문제 08 | 문자열 처리

`Name` 컬럼에서 **호칭(Title)** 을 추출하세요.

- 예: `"Braund, Mr. Owen Harris"` → `Mr`  
- 힌트: `str.extract(r',\s*([^\.]+)\.')` 또는 `str.split`
- 새 컬럼 `Title` 생성 후 `value_counts()` 상위 5개 출력

In [10]:
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.").squeeze()
df["Title"].value_counts().head()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Dr          7
Name: count, dtype: int64

---
## 문제 09 | lambda + apply

`Age` 컬럼에 `apply` + `lambda`를 사용해 `AgeGroup` 컬럼을 만드세요.

| 조건 | AgeGroup |
|------|----------|
| 결측 | `'Unknown'` |
| 18 미만 | `'Child'` |
| 18 ~ 59 | `'Adult'` |
| 60 이상 | `'Senior'` |

`AgeGroup`별 승객 수를 `value_counts()`로 출력하세요.

In [11]:
df["AgeGroup"] = df["Age"].apply(
    lambda x: "Unknown" if pd.isna(x)
    else "Child" if x < 18
    else "Adult" if x < 60
    else "Senior"
)
df["AgeGroup"].value_counts()

AgeGroup
Adult      575
Unknown    177
Child      113
Senior      26
Name: count, dtype: int64

---
## 문제 10 | numpy vectorization

반복문/`apply` 없이 **numpy 벡터 연산**만으로 아래를 수행하세요.

1. `FamilySize = SibSp + Parch + 1` (`.values` 또는 numpy 배열 활용)  
2. `IsAlone = (FamilySize == 1)` 불리언 배열  
3. `np.where`로 `FareLevel` 생성: Fare ≥ 30 → `'High'`, 아니면 `'Low'`  
4. **혼자 탑승(IsAlone)이면서 High 요금**인 승객 수 출력

In [12]:
family_size = df["SibSp"].values + df["Parch"].values + 1
is_alone = family_size == 1
fare_level = np.where(df["Fare"].values >= 30, "High", "Low")

alone_high_count = np.sum(is_alone & (fare_level == "High"))
print("혼자 탑승 + High 요금 승객 수:", alone_high_count)

혼자 탑승 + High 요금 승객 수: 79
